In [ ]:
!pip install -q -U gspread openai

In [ ]:
from google.colab import userdata, auth
import google.auth
import gspread
from openai import OpenAI

auth.authenticate_user()

creds, _ = google.auth.default()
sheets_client = gspread.authorize(creds)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY")
)

print("Google Sheets and LLM connected.")

In [ ]:
spreadsheet = sheets_client.open_by_key(
    "YOUR_GOOGLE_SHEET_ID"
)

print("Google Sheet connected.")

In [ ]:
def number(value):
    try:
        return float(
            str(value).replace(",", "").replace("%", "").strip() or 0
        )
    except:
        return 0


def records(sheet_name):
    return spreadsheet.worksheet(sheet_name).get_all_records()


def get_attendance(employee_id):
    return [
        r for r in records("Attendance")
        if r["Employee ID"] == employee_id
    ]


def get_sales(employee_id):
    return [
        r for r in records("Sales")
        if r["Employee ID"] == employee_id
    ]


def get_attendance_policy():
    return records("Attendance_Policy")


def get_commission_policy():
    return records("Commission_Policy")


def get_payroll(employee_id=None):
    data = records("Payroll")

    return [
        r for r in data
        if not employee_id or r["Employee ID"] == employee_id
    ]


def active_policy(policy, policy_type, day_type=None):
    result = [
        r for r in policy
        if r["Policy Type"] == policy_type
        and str(r["Active"]).strip().lower() == "yes"
    ]

    if day_type:
        result = [
            r for r in result
            if r["Day Type"] == day_type
        ]

    return result


def attendance_pay(employee_id):
    attendance = get_attendance(employee_id)
    policy = get_attendance_policy()

    result = {
        "currency": policy[0]["Currency"] if policy else "",
        "overtime": 0,
        "late_entry": 0,
        "early_exit": 0,
        "violations": []
    }

    for row in attendance:

        day_type = row["Day Type"]

        late = active_policy(
            policy,
            "Late Entry",
            day_type
        )

        early = active_policy(
            policy,
            "Early Exit",
            day_type
        )

        overtime = active_policy(
            policy,
            "Overtime",
            day_type
        )

        if late:
            p = late[0]

            result["late_entry"] += (
                number(row["Late Entry Hours"])
                * number(p["Hourly Rate"])
                * number(p["Multiplier"])
            )

        if early:
            p = early[0]

            result["early_exit"] += (
                number(row["Early Exit Hours"])
                * number(p["Hourly Rate"])
                * number(p["Multiplier"])
            )

        if overtime:
            p = overtime[0]

            hours = number(row["Overtime Hours"])

            minimum = number(
                p["Daily Minimum Payable Hours"]
            )

            maximum = number(
                p["Daily Maximum Payable Hours"]
            )

            rate = number(
                p["Hourly Rate"]
            )

            multiplier = number(
                p["Multiplier"]
            )

            if hours >= minimum:

                payable_hours = min(
                    hours,
                    maximum
                )

                result["overtime"] += (
                    payable_hours
                    * rate
                    * multiplier
                )

                if hours > maximum:
                    result["violations"].append({
                        "date": row["Attendance Date"],
                        "day_type": day_type,
                        "actual_hours": hours,
                        "payable_hours": maximum,
                        "excess_hours": hours - maximum
                    })

    return result


def commission_pay(employee_id):
    sales = get_sales(employee_id)
    policy = get_commission_policy()

    total_sales = sum(
        number(row["Sales Amount"])
        for row in sales
    )

    for p in policy:

        minimum = number(
            p["From Sales Amount"]
        )

        maximum = number(
            p["To Sales Amount"]
        )

        if minimum <= total_sales <= maximum:

            rate = number(
                p["Commission Percent"]
            )

            return {
                "sales": total_sales,
                "commission": total_sales * rate / 100,
                "rate": rate,
                "calculation_type": p["Calculation Type"]
            }

    return {
        "sales": total_sales,
        "commission": 0,
        "rate": 0,
        "calculation_type": ""
    }


def verify_payroll(employee_id):
    payroll = get_payroll(employee_id)

    if not payroll:
        return {
            "error": "Payroll record not found."
        }

    row = payroll[0]

    attendance = attendance_pay(employee_id)
    commission = commission_pay(employee_id)

    expected = {
        "commission": commission["commission"],

        "overtime": attendance["overtime"],

        "total_earnings": (
            commission["commission"]
            + attendance["overtime"]
        ),

        "late_deduction": attendance["late_entry"],

        "early_deduction": attendance["early_exit"],

        "total_deductions": (
            attendance["late_entry"]
            + attendance["early_exit"]
        )
    }

    actual = {
        "commission": number(
            row["Commission Earning"]
        ),

        "overtime": number(
            row["Overtime Earning"]
        ),

        "total_earnings": number(
            row["Total Earnings"]
        ),

        "late_deduction": number(
            row["Late Entry Deduction"]
        ),

        "early_deduction": number(
            row.get(
                "Early Exit Dedcution",
                row.get(
                    "Early Exit Deduction",
                    0
                )
            )
        ),

        "total_deductions": number(
            row["Total Deductions"]
        )
    }

    differences = {
        key: actual[key] - expected[key]
        for key in expected
    }

    return {
        "employee_id": employee_id,

        "employee": row["Employee"],

        "salary_slip_id": row["Salary Slip ID"],

        "currency": attendance["currency"],

        "payroll": actual,

        "expected": expected,

        "differences": differences,

        "commission_sales": commission["sales"],

        "commission_rate": commission["rate"],

        "overtime_violations": attendance["violations"],

        "earnings_correct": all(
            actual[key] == expected[key]
            for key in [
                "commission",
                "overtime",
                "total_earnings"
            ]
        ),

        "deductions_correct": all(
            actual[key] == expected[key]
            for key in [
                "late_deduction",
                "early_deduction",
                "total_deductions"
            ]
        )
    }


def update_payroll(employee_id, approved=False):

    if not approved:
        return {
            "status": "Update blocked",
            "message": "Human approval is required."
        }

    result = verify_payroll(employee_id)

    if "error" in result:
        return result

    sheet = spreadsheet.worksheet("Payroll")
    data = sheet.get_all_records()

    for i, row in enumerate(data, start=2):

        if row["Employee ID"] == employee_id:

            expected = result["expected"]

            sheet.update(
                f"G{i}:L{i}",
                [[
                    expected["commission"],
                    expected["overtime"],
                    expected["total_earnings"],
                    expected["late_deduction"],
                    expected["early_deduction"],
                    expected["total_deductions"]
                ]]
            )

            return {
                "status": "Updated",
                "employee_id": employee_id
            }

    return {
        "error": "Payroll row not found."
    }

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_attendance",
            "description": "Get attendance records for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string"
                    }
                },
                "required": ["employee_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_sales",
            "description": "Get sales records for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string"
                    }
                },
                "required": ["employee_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_attendance_policy",
            "description": "Get current attendance policies from Google Sheets.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_commission_policy",
            "description": "Get commission policies from Google Sheets.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "verify_payroll",
            "description": "Verify an employee payroll entry against attendance, sales and policy data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string"
                    }
                },
                "required": ["employee_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "update_payroll",
            "description": "Update payroll only after explicit human approval.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string"
                    },
                    "approved": {
                        "type": "boolean"
                    }
                },
                "required": ["employee_id", "approved"]
            }
        }
    }
]

In [ ]:
tool_registry = {
    "get_attendance": get_attendance,
    "get_sales": get_sales,
    "get_attendance_policy": get_attendance_policy,
    "get_commission_policy": get_commission_policy,
    "verify_payroll": verify_payroll,
    "update_payroll": update_payroll
}

print("Agent tools registered.")

Agent tools registered.


In [ ]:
SYSTEM_PROMPT = """
You are a goal-driven AI Payroll Verification Agent.

Understand the user's goal and select only the tools needed.
Do not follow a fixed workflow.

Google Sheets is the source of truth.

Python tool results are authoritative.

Never:
- invent data
- invent policy
- recalculate numbers
- change numbers returned by Python
- make unsupported assumptions
- speculate about why a payroll value is wrong

When verification results contain a "differences" field, use those
values exactly as provided by Python.

Do not calculate differences yourself.

For overtime:
- Minimum Hours = minimum hours required for payment eligibility.
- Maximum Hours = maximum hours that can be paid.
- If actual hours exceed maximum, only the maximum payable hours
  are included in expected pay.
- Excess hours are not included in expected pay.
- Report excess hours only when Python identifies them.
- Do not call excess hours a violation unless Python identifies
  them as an issue.

For commission:
- Use the commission result returned by Python.
- Do not independently calculate commission.
- Do not invent or assume why the payroll commission is different.

For deductions:
- Use the expected deduction values returned by Python.
- Do not assume there are other deductions unless Python provides
  them.
- Do not invent reasons for deduction differences.

When explaining an incorrect amount:
- State the payroll value.
- State the expected value.
- State the exact Python-provided difference.
- Explain only what is supported by the Python result.
- If the reason is not available, say:
  "The available data does not provide enough information."

Never use words such as:
- presumably
- probably
- likely
- maybe

unless they are explicitly part of the data returned by Python.

Clearly explain:
- payroll value
- expected value
- Python-provided difference
- status
- supported reason, if available

Do not update payroll automatically.

Payroll updates require explicit human approval.
"""


def run_agent(goal):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": goal}
    ]

    while True:
        response = client.chat.completions.create(
            model="openrouter/free",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        if not message.tool_calls:
            print(message.content)
            break

        messages.append(message)

        for call in message.tool_calls:
            name = call.function.name
            args = eval(call.function.arguments)

            result = tool_registry[name](**args)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result)
            })

In [ ]:
def approve_payroll(employee_id):
    result = verify_payroll(employee_id)

    if "error" in result:
        print(result["error"])
        return

    print("Payroll Verification")
    print("--------------------")
    print(f"Employee: {result['employee']}")
    print(f"Salary Slip: {result['salary_slip_id']}")
    print()

    print("Current Payroll:")
    print(result["payroll"])

    print("\nExpected Payroll:")
    print(result["expected"])

    print("\nDifferences:")
    print(result["differences"])

    approval = input("\nApprove these corrections? (yes/no): ").strip().lower()

    if approval == "yes":
        result = update_payroll(employee_id, approved=True)
        print("\n", result)
    else:
        print("\nUpdate cancelled. No changes were made.")

In [ ]:
approve_payroll("EMP001")